# Cart2Insights — Feature Engineering

## Objective

Create business-oriented analytical features from the cleaned data
stored in PostgreSQL.

The official project brief requires:

- Total order value
- Delivery days
- Delivery delay
- Customer order count
- Customer total spending
- Average order value
- Seller revenue
- Seller order count
- Repeat customer indicator

## Feature Engineering Principle

Before creating a feature, define:

1. Business purpose
2. Grain
3. Source table(s)
4. Transformation/formula
5. Assumptions
6. Validation

Because the project contains multiple one-to-many relationships, child
tables such as `order_items`, `order_payments`, and `order_reviews`
must not be joined directly when calculating order-level metrics.
Order-level aggregates should be created first to avoid row multiplication.

In [ ]:
from pathlib import Path
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

In [ ]:
PROJECT_ROOT = Path.cwd().parent

load_dotenv(PROJECT_ROOT / ".env")

DB_HOST = os.getenv("POSTGRES_HOST")
DB_PORT = os.getenv("POSTGRES_PORT")
DB_USER = os.getenv("POSTGRES_USER")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD")
DB_NAME = os.getenv("POSTGRES_DATABASE")

DATABASE_URL = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=int(DB_PORT),
    database=DB_NAME,
)

engine = create_engine(DATABASE_URL)

In [ ]:
PROJECT_ROOT = Path.cwd().parent

load_dotenv(PROJECT_ROOT / ".env")

DB_HOST = os.getenv("POSTGRES_HOST")
DB_PORT = os.getenv("POSTGRES_PORT")
DB_USER = os.getenv("POSTGRES_USER")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD")
DB_NAME = os.getenv("POSTGRES_DATABASE")

DATABASE_URL = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=int(DB_PORT),
    database=DB_NAME,
)

engine = create_engine(DATABASE_URL)

In [ ]:
with engine.connect() as connection:
    connection.execute(text("SELECT 1"))

print("PostgreSQL connection successful!")

In [ ]:
orders = pd.read_sql(
    """
    SELECT
        order_id,
        customer_id,
        order_status,
        order_purchase_timestamp,
        order_delivered_customer_date,
        order_estimated_delivery_date
    FROM orders
    """,
    engine,
)

order_items = pd.read_sql(
    """
    SELECT
        order_id,
        order_item_id,
        product_id,
        seller_id,
        price,
        freight_value
    FROM order_items
    """,
    engine,
)

customers = pd.read_sql(
    """
    SELECT
        customer_id,
        customer_unique_id
    FROM customers
    """,
    engine,
)


sellers = pd.read_sql(
    """
    SELECT
        seller_id
    FROM sellers
    """,
    engine,
)

In [ ]:
orders.dtypes

# Feature 1 — Total Order Value

## Business Question

What is the total monetary value associated with each order?

## Why do we need this?

The `order_items` table stores monetary values at item level:

- `price`
- `freight_value`

The business analysis, however, often needs the value at order level.

Therefore, we transform item-level information into an order-level
feature.

## Grain

Input grain:
One row = one order item

Output grain:
One row = one order

## Definition

For this project, we define:

`item_total_value = price + freight_value`

Then:

`total_order_value = sum(item_total_value for all items in the order)`

## Important Join Decision

Only `order_items` is required for this calculation.

We do not join payments or reviews here because those tables also contain
multiple rows per order and could multiply the order-item rows.

In [ ]:
order_items["item_total_value"] = (
    order_items["price"]
    + order_items["freight_value"]
)

order_items[
    [
        "order_id",
        "price",
        "freight_value",
        "item_total_value",
    ]
].head()

In [ ]:
order_values = (
    order_items
    .groupby("order_id", as_index=False)
    .agg(
        total_order_value=("item_total_value", "sum")
    )
)

In [ ]:
order_values.head(10)

In [ ]:
assert order_values["order_id"].is_unique

assert (
    order_values["total_order_value"] >= 0
).all()

print("Order IDs are unique:", order_values["order_id"].is_unique)
print(
    "Negative total order values:",
    (order_values["total_order_value"] < 0).sum()
)

# Feature 2 — Delivery Days

## Business Question

How long did the customer actually wait from placing the order until
the order was delivered?

## Definition

`delivery_days = order_delivered_customer_date - order_purchase_timestamp`

## Grain

Output grain:
One row = one order

## Missing Values

If `order_delivered_customer_date` is missing, the actual delivery duration
cannot be calculated. We will keep the feature as missing rather than
replacing it with zero or another invented value.

## Business Use

This feature will later support:

- delivery performance analysis
- delivery distributions and trends
- comparison of delivery time across locations
- customer satisfaction analysis
- the required statistical analysis

In [ ]:
orders.dtypes

In [ ]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

In [ ]:
orders[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "delivery_days",
    ]
].head(10)

In [ ]:
valid_delivery_days = orders["delivery_days"].dropna()

print("Orders:", len(orders))
print("Orders with delivery_days:", len(valid_delivery_days))
print("Missing delivery_days:", orders["delivery_days"].isna().sum())

print("\nDelivery days summary:")
print(valid_delivery_days.describe())

assert (valid_delivery_days >= 0).all()

In [ ]:
negative_delivery_days = (
    orders["delivery_days"] < 0
).sum()

print("Negative delivery_days:", negative_delivery_days)

assert negative_delivery_days == 0

# Feature 3 — Delivery Delay

## Business Question

Did the order arrive later or earlier than the estimated delivery date?

## Definition

`delivery_delay = actual_delivery_date - estimated_delivery_date`

The result is measured in days.

Interpretation:

- Positive value → delivered after the estimated date
- Negative value → delivered before the estimated date
- Zero → delivered exactly on the estimated date

## Grain

Output grain:
One row = one order

## Missing Values

If the order has no actual delivery date, delivery delay cannot be
calculated and will remain missing.

## Why is this different from `delivery_days`?

`delivery_days` measures the customer's actual waiting time.

`delivery_delay` measures performance against the promised/estimated
delivery date.

An order can have a long delivery duration but still be on time if the
estimated date was sufficiently far in the future.

Conversely, an order can have a relatively short delivery duration and
still be delayed if it arrived later than promised.

In [ ]:
orders["delivery_delay"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 60 * 60)

In [ ]:
orders["delivery_status"] = pd.NA

delivered_mask = orders["order_delivered_customer_date"].notna()

orders.loc[
    delivered_mask & (orders["delivery_delay"] > 0),
    "delivery_status"
] = "Delayed"

orders.loc[
    delivered_mask & (orders["delivery_delay"] <= 0),
    "delivery_status"
] = "On Time"

In [ ]:
orders[
    [
        "order_id",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_delay",
        "delivery_status",
    ]
].head(10)

In [ ]:
print(
    orders["delivery_status"]
    .value_counts(dropna=False)
)

In [ ]:
print(
    "Missing delivery delay:",
    orders["delivery_delay"].isna().sum()
)

print(
    "Missing actual delivery date:",
    orders["order_delivered_customer_date"].isna().sum()
)

assert (
    orders["delivery_delay"].isna()
    ==
    orders["order_delivered_customer_date"].isna()
).all()

In [ ]:
delivered_statuses = orders.loc[
    orders["order_delivered_customer_date"].notna(),
    "delivery_status"
]

assert delivered_statuses.notna().all()
assert set(delivered_statuses.unique()) <= {
    "Delayed",
    "On Time",
}

### Interpretation

`delivery_delay` measures delivery performance relative to the estimated
delivery date.

- Negative values indicate the order arrived earlier than estimated.
- Zero indicates delivery on the estimated date.
- Positive values indicate the order arrived after the estimated date.
- Missing values indicate that an actual delivery date is unavailable.

`delivery_status` converts this numerical measure into a business-friendly
classification:

- `On Time` → `delivery_delay <= 0`
- `Delayed` → `delivery_delay > 0`

The feature will later support delivery-performance analysis,
customer-satisfaction analysis, and the required statistical comparison
of review scores between delayed and on-time orders.

# Feature 4 — Customer Order Count

## Business Question
How many distinct orders has each customer placed?

## Customer Identity
For customer behavior analysis, we use `customer_unique_id`
as the customer-level analytical identifier.

`customer_id` is the relational key linking an order to a customer
record, while `customer_unique_id` allows us to identify the same
customer across multiple customer records and orders.

## Grain
Input:
- `orders`: one row = one order
- `customers`: one row = one customer record

Output:
- one row = one `customer_unique_id`

## Definition
`customer_order_count` is the number of distinct `order_id` values
associated with each `customer_unique_id`.

## Why do we need it?
This feature measures customer purchase frequency and will later
be used to identify repeat customers and calculate customer-level
average order value.

## Transformation Logic
1. Map each order's `customer_id` to `customer_unique_id`.
2. Group orders by `customer_unique_id`.
3. Count distinct `order_id` values for each customer.

## Assumption
The mapping from `customer_id` to `customer_unique_id` should be
many-to-one: multiple customer records may belong to the same
analytical customer, but each `customer_id` should map to one
`customer_unique_id`.

## Validation
We will verify:
- no missing `customer_unique_id` after the mapping
- one output row per `customer_unique_id`
- every customer has at least one order
- order counts are positive
- the distribution of order counts is plausible

In [ ]:
orders_customer = orders[
    ["order_id", "customer_id"]
].merge(
    customers[
        ["customer_id", "customer_unique_id"]
    ],
    on="customer_id",
    how="left",
    validate="many_to_one"
)

orders_customer.head()

In [ ]:
customer_order_count = (
    orders_customer
    .groupby("customer_unique_id", as_index=False)
    .agg(
        customer_order_count=("order_id", "nunique")
    )
)

customer_order_count.head()

In [ ]:
assert orders_customer["customer_unique_id"].notna().all()
assert customer_order_count["customer_unique_id"].is_unique
assert (customer_order_count["customer_order_count"] >= 1).all()

print("Orders represented:", len(orders_customer))
print(
    "Unique customer identities:",
    customer_order_count["customer_unique_id"].nunique()
)

print(
    "\nCustomer order-count distribution:"
)

print(
    customer_order_count["customer_order_count"]
    .value_counts()
    .sort_index()
)

# Feature 5 — Customer Total Spending

## Business Question
How much total recorded order value is associated with each customer?

## Customer Identity
Customer behavior is analyzed at the `customer_unique_id` grain.

## Grain
Input:
- `order_values`: one row = one order
- customer mapping: one row = one order after mapping

Output:
- one row = one `customer_unique_id`

## Definition
`customer_total_spending` is the sum of `total_order_value`
across orders associated with the same `customer_unique_id`.

## Order Status Treatment
The feature uses all orders with a recorded `total_order_value`
rather than restricting the calculation to delivered orders.

The project brief requires customer total spending but does not
define it as delivered-only spending. Therefore, we do not introduce
a delivered-order filter at this stage.

Order status remains available as a separate analytical variable
for later sales, delivery, and operational analysis.

## Transformation Logic
1. Start from the order-level `total_order_value`.
2. Map each order to `customer_unique_id`.
3. Group by `customer_unique_id`.
4. Sum `total_order_value`.

## Important Grain Consideration
`order_values` is already aggregated to order grain before the
customer aggregation. This prevents multiple order-item rows from
being incorrectly summed as separate customer-level records.

## Validation
We will verify:
- every order has a mapped customer identity
- the order-level merge preserves 99,441 orders
- one output row exists per `customer_unique_id`
- spending values are non-negative
- customer spending totals reconcile with order-level values

In [ ]:
customer_order_values = orders_customer[
    ["order_id", "customer_unique_id"]
].merge(
    order_values,
    on="order_id",
    how="left",
    validate="one_to_one"
)

customer_order_values.head()

In [ ]:
customer_total_spending = (
    customer_order_values
    .groupby("customer_unique_id", as_index=False)
    .agg(
        customer_total_spending=("total_order_value", "sum")
    )
)

customer_total_spending.head()

In [ ]:
assert customer_order_values["customer_unique_id"].notna().all()
assert len(customer_order_values) == len(orders)
assert customer_total_spending["customer_unique_id"].is_unique
assert customer_total_spending["customer_total_spending"].notna().all()
assert (customer_total_spending["customer_total_spending"] >= 0).all()

print("Orders represented:", len(customer_order_values))
print(
    "Unique customers represented:",
    customer_total_spending["customer_unique_id"].nunique()
)

print("\nCustomer spending summary:")
print(customer_total_spending["customer_total_spending"].describe())

In [ ]:
order_value_total = order_values["total_order_value"].sum()
customer_value_total = (
    customer_total_spending["customer_total_spending"].sum()
)

print("Order-level total:", order_value_total)
print("Customer-level total:", customer_value_total)

print(
    "Totals reconcile:",
    abs(order_value_total - customer_value_total) < 1e-9
)

# Feature 6 — Customer Average Order Value

## Business Question
What is the average recorded order value for each customer?

## Grain
Input:
- one row = one `customer_unique_id`

Output:
- one row = one `customer_unique_id`

## Definition
`customer_average_order_value` =
`customer_total_spending / customer_order_count`

## Why do we need it?
Customer total spending tells us how much a customer has spent overall,
while customer average order value tells us the typical value of each
order placed by that customer.

This helps distinguish frequent lower-value customers from customers
who place fewer but higher-value orders.

## Assumption
Every represented customer has at least one recorded order, so the
order count used as the denominator is greater than zero.

## Downstream Use
This feature can support customer behavior analysis, segmentation,
and comparison of purchasing patterns.

In [ ]:
customer_features = customer_order_count.merge(
    customer_total_spending,
    on="customer_unique_id",
    how="inner",
    validate="one_to_one"
)

customer_features["customer_average_order_value"] = (
    customer_features["customer_total_spending"]
    / customer_features["customer_order_count"]
)

customer_features.head()

In [ ]:
assert customer_features["customer_unique_id"].is_unique
assert customer_features["customer_order_count"].gt(0).all()
assert customer_features["customer_average_order_value"].notna().all()
assert customer_features["customer_average_order_value"].ge(0).all()

print("Customers represented:", len(customer_features))
print(
    "Unique customer identities:",
    customer_features["customer_unique_id"].nunique()
)

print("\nCustomer AOV summary:")
print(
    customer_features["customer_average_order_value"].describe()
)

In [ ]:
formula_check = (
    customer_features["customer_average_order_value"]
    ==
    customer_features["customer_total_spending"]
    / customer_features["customer_order_count"]
)

print("Formula valid:", formula_check.all())

# Feature 7 — Repeat Customer Indicator

## Business Question
Which customers have placed more than one recorded order?

## Grain
Input:
- one row = one `customer_unique_id`

Output:
- one row = one `customer_unique_id`

## Definition
A customer is classified as:
- `Repeat` when `customer_order_count > 1`
- `One-time` when `customer_order_count == 1`

## Why do we need it?
This feature converts purchase frequency into an interpretable
customer-behavior indicator.

It supports repeat-vs-new customer analysis and customer
segmentation.

## Source
The feature is derived from the validated
`customer_order_count` feature.

## Downstream Use
It will be used for customer behavior analysis and dashboard
reporting.

In [ ]:
customer_features["repeat_customer"] = (
    customer_features["customer_order_count"] > 1
)

customer_features["customer_type"] = (
    customer_features["repeat_customer"]
    .map({
        True: "Repeat",
        False: "One-time"
    })
)

customer_features.head()

In [ ]:
assert customer_features["customer_unique_id"].is_unique
assert customer_features["repeat_customer"].notna().all()
assert customer_features["customer_type"].notna().all()

print(
    customer_features["customer_type"]
    .value_counts()
)

In [ ]:
repeat_check = (
    customer_features["repeat_customer"]
    ==
    (customer_features["customer_order_count"] > 1)
)

print("Repeat indicator valid:", repeat_check.all())

In [ ]:
customer_type_summary = (
    customer_features["customer_type"]
    .value_counts()
    .rename_axis("customer_type")
    .reset_index(name="customer_count")
)

customer_type_summary["percentage"] = (
    customer_type_summary["customer_count"]
    / customer_type_summary["customer_count"].sum()
    * 100
)

customer_type_summary

# Feature 8 — Seller Revenue

## Business Question
What is the total recorded product sales value associated with each seller?

## Grain
Input:
- `order_items`: one row = one order item

Output:
- one row = one `seller_id`

## Definition
`seller_revenue` is the sum of `price` across all order-item
records belonging to a seller.

## Why do we need it?
This feature measures seller-level sales performance and supports
ranking sellers by their recorded product sales value.

## Freight Treatment
`freight_value` is not included in `seller_revenue`.

The feature uses `price` as the product sales amount and keeps
freight as a separate logistics-related measure.

The official brief requires seller revenue but does not provide
an explicit accounting definition, so this is an analytical
definition chosen from the available order-item fields.

## Transformation Logic
1. Start with `order_items`.
2. Group by `seller_id`.
3. Sum `price`.

## Grain Consideration
No join is required because `seller_id` and `price` already exist
at the order-item grain.

## Validation
We will verify:
- one row per seller
- all seller IDs are represented where applicable
- revenue is non-negative
- seller revenue reconciles with total item price

In [ ]:
seller_revenue = (
    order_items
    .groupby("seller_id", as_index=False)
    .agg(
        seller_revenue=("price", "sum")
    )
)

seller_revenue.head()

In [ ]:
assert seller_revenue["seller_id"].is_unique
assert seller_revenue["seller_revenue"].notna().all()
assert (seller_revenue["seller_revenue"] >= 0).all()

print("Sellers represented:", len(seller_revenue))
print(
    "Unique seller IDs:",
    seller_revenue["seller_id"].nunique()
)

print("\nSeller revenue summary:")
print(seller_revenue["seller_revenue"].describe())

In [ ]:
item_price_total = order_items["price"].sum()
seller_revenue_total = seller_revenue["seller_revenue"].sum()

print("\nOrder-item price total:", item_price_total)
print("Seller revenue total:", seller_revenue_total)

# Feature 9 — Seller Order Count

## Business Question
How many distinct orders did each seller participate in?

## Grain
Input:
- `order_items`: one row = one order item

Output:
- one row = one `seller_id`

## Definition
`seller_order_count` is the number of distinct `order_id`
values associated with each seller.

## Why do we need it?
This feature measures seller order volume and complements
`seller_revenue` for seller performance analysis.

## Why count distinct orders?
An order can contain multiple items from the same seller.
Counting order-item rows would therefore overstate the number
of orders handled by that seller.

## Transformation Logic
1. Start from `order_items`.
2. Group by `seller_id`.
3. Count distinct `order_id` values.

## Validation
We will verify:
- one row per seller
- order counts are positive
- seller IDs are represented correctly
- the result does not exceed the number of distinct platform orders

In [ ]:
seller_order_count = (
    order_items
    .groupby("seller_id", as_index=False)
    .agg(
        seller_order_count=("order_id", "nunique")
    )
)

seller_order_count.head()

In [ ]:
assert seller_order_count["seller_id"].is_unique
assert seller_order_count["seller_order_count"].notna().all()
assert (seller_order_count["seller_order_count"] >= 1).all()

print("Sellers represented:", len(seller_order_count))
print(
    "Unique seller IDs:",
    seller_order_count["seller_id"].nunique()
)

print("\nSeller order-count summary:")
print(
    seller_order_count["seller_order_count"].describe()
)

In [ ]:
print(
    "Maximum seller order count:",
    seller_order_count["seller_order_count"].max()
)

print(
    "Distinct platform orders:",
    order_items["order_id"].nunique()
)

In [ ]:
seller_features = seller_revenue.merge(
    seller_order_count,
    on="seller_id",
    how="inner",
    validate="one_to_one"
)

print("Seller feature rows:", len(seller_features))
print("Unique sellers:", seller_features["seller_id"].nunique())

seller_features.head()

# Final Order Feature Dataset

## Grain
One row = one `order_id`

## Included Features
- `total_order_value`
- `delivery_days`
- `delivery_delay`
- `delivery_status`

The dataset is assembled only from previously validated
order-level features.

In [ ]:
order_features = orders[
    ["order_id", "delivery_days", "delivery_delay", "delivery_status"]
].merge(
    order_values,
    on="order_id",
    how="left",
    validate="one_to_one"
)

order_features = order_features[
    [
        "order_id",
        "total_order_value",
        "delivery_days",
        "delivery_delay",
        "delivery_status",
    ]
]

order_features.head()

In [ ]:
assert order_features["order_id"].is_unique
assert len(order_features) == len(orders)

print("Order feature rows:", len(order_features))
print("Unique order IDs:", order_features["order_id"].nunique())

# Final Customer Feature Dataset

## Grain
One row = one `customer_unique_id`

## Included Features
- `customer_order_count`
- `customer_total_spending`
- `customer_average_order_value`
- `repeat_customer`
- `customer_type`

The customer-level features use `customer_unique_id` because
customer behavior must be measured across customer records and orders.

In [ ]:
customer_features = customer_features[
    [
        "customer_unique_id",
        "customer_order_count",
        "customer_total_spending",
        "customer_average_order_value",
        "repeat_customer",
        "customer_type",
    ]
]

customer_features.head()

In [ ]:
assert customer_features["customer_unique_id"].is_unique
assert customer_features["customer_total_spending"].ge(0).all()
assert customer_features["customer_average_order_value"].ge(0).all()

print("Customer feature rows:", len(customer_features))
print(
    "Unique customer IDs:",
    customer_features["customer_unique_id"].nunique()
)

# Final Seller Feature Dataset

## Grain
One row = one `seller_id`

## Included Features
- `seller_revenue`
- `seller_order_count`

These features summarize seller-level sales value and order volume.

In [ ]:
seller_features = seller_features[
    [
        "seller_id",
        "seller_revenue",
        "seller_order_count",
    ]
]

seller_features.head()

In [ ]:
assert seller_features["seller_id"].is_unique
assert seller_features["seller_revenue"].ge(0).all()
assert seller_features["seller_order_count"].ge(1).all()

print("Seller feature rows:", len(seller_features))
print(
    "Unique seller IDs:",
    seller_features["seller_id"].nunique()
)

In [ ]:
from pathlib import Path

FEATURES_DIR = PROJECT_ROOT / "data" / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

order_features.to_csv(
    FEATURES_DIR / "order_features.csv",
    index=False
)

customer_features.to_csv(
    FEATURES_DIR / "customer_features.csv",
    index=False
)

seller_features.to_csv(
    FEATURES_DIR / "seller_features.csv",
    index=False
)

print("Feature datasets saved successfully.")

In [ ]:
feature_files = [
    FEATURES_DIR / "order_features.csv",
    FEATURES_DIR / "customer_features.csv",
    FEATURES_DIR / "seller_features.csv",
]

for file_path in feature_files:
    print(file_path, "→", "Exists" if file_path.exists() else "Missing")

In [ ]:
saved_order_features = pd.read_csv(
    FEATURES_DIR / "order_features.csv"
)

saved_customer_features = pd.read_csv(
    FEATURES_DIR / "customer_features.csv"
)

saved_seller_features = pd.read_csv(
    FEATURES_DIR / "seller_features.csv"
)

print("Order features:", saved_order_features.shape)
print("Customer features:", saved_customer_features.shape)
print("Seller features:", saved_seller_features.shape)